_Neural Data Science_

Lecturer: Prof. Dr. Philipp Berens, Dr. Jan Lause

Tutors: Jonas Beck, Kyra Kadhim, Jonathan Oesterle, Julius Würzler

Summer term 2026

Student names: <span style='background: yellow'>Frauke von der Haar, Urmi Jana </span>

LLM Disclaimer: <span style='background: yellow'>*Did you use an LLM to solve this exercise? If yes, which one and where did you use it? [Copilot, Claude, ChatGPT, etc.]* </span>

# Neural Data Science Project — Spike Inference

## Inferring spiking activity from calcium imaging data using deep learning

In this project you will train your own deep network to infer neuronal spiking activity from calcium imaging ΔF/F traces. You will work with ground truth data from simultaneous calcium imaging and electrophysiology recordings, preprocess it for training, build and train a neural network, and evaluate your model on held-out neurons.

You are free to use tools, resources and libraries as you see fit. Use comments and markdown cells to document your thought process and to explain your reasoning. We encourage you to compare different algorithms or to implement state of the art solutions. The notebook should be self contained, although you may offload some functions to a `utils.py`. The notebook should be concluded with a final summary / conclusions section.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.signal import resample
from scipy.stats import pearsonr

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

## Context

### Calcium imaging and spike inference

Two-photon calcium imaging records neural activity by measuring fluorescence of genetically encoded calcium indicators (e.g. GCaMP6f, GCaMP6s). When a neuron fires an action potential, calcium flows into the cell, causing the indicator to fluoresce more brightly. The resulting signal — expressed as **ΔF/F** (relative fluorescence change) — is a noisy, temporally smoothed version of the true spiking activity:

- A single action potential produces a fast rise (~50 ms) followed by a slow exponential decay (~200–1000 ms, depending on the indicator)
- The signal is corrupted by shot noise, neuropil contamination, and motion artifacts
- Multiple spikes in quick succession sum nonlinearly

**Spike inference** (also called deconvolution) is the inverse problem of recovering the underlying spike times or spike rates from the observed ΔF/F traces.

### Cascade as a reference

In this project you will train your own neural network for the task of spike inference, similar to what was done in [**Cascade**](https://github.com/HelmchenLabSoftware/Cascade) (Rupprecht et al., *Nature Neuroscience*, 2021). Cascade trains a simple 1D convolutional network (~50k parameters) on a large database of simultaneous calcium imaging and electrophysiology recordings. A key part of their approach is a noise-matched training procedure that resamples the ground truth data to match the noise level and frame rate of any target recording, making the trained models robust across different experimental conditions. Cascade also provides pretrained models for common configurations that can serve as performance baselines.

You can use their code, preprocessing routines, and pretrained models as a reference. The PyTorch implementation is available at [CascadeTorch](https://github.com/PTRRupprecht/CascadeTorch).

**Paper:** Rupprecht P, Carta S, Hoffmann A, Echizen M, Blot A, Kwan AC, Dan Y, Hofer SB, Kitamura K, Helmchen F\*, Friedrich RW\*. *A database and deep learning toolbox for noise-optimized, generalized spike inference from calcium imaging.* Nature Neuroscience (2021). [Link](https://www.nature.com/articles/s41593-021-00895-5)

### The data

The file `spike_inference_data.npz` contains ground truth recordings from simultaneous calcium imaging and electrophysiology. Each recording provides:

| Field | Description |
|---|---|
| `dff` | ΔF/F fluorescence trace (1D float array) |
| `t` | Time vector in seconds (1D float array) |
| `spikes` | Action potential times in seconds (1D float array) |
| `frame_rates` | Imaging frame rate in Hz (scalar) |
| `datasets` | Source dataset identifier (string) |

The data is split into:

- **Training recordings** (`train_*`): Use these to build your preprocessing pipeline and train your network. They come from 4 ground truth datasets recorded with GCaMP6f and GCaMP6s in mouse visual cortex.
- **Test recordings** (`test_*`): A held-out dataset for final evaluation. Do not use these for training.

All recordings were obtained at ~60 Hz and can be resampled to a lower frame rate (e.g. 30 Hz) during preprocessing.

In [ ]:
# load data
def load_data(path="./data"):
    data = np.load(path + "/spike_inference_data.npz", allow_pickle=True)
    return dict(data)

def print_info(data):
    for key in sorted(data.keys()):
        arr = data[key]
        if arr.dtype == object:
            print(f"  [{key:25s}]  {arr.shape}  (variable-length arrays)")
        else:
            print(f"  [{key:25s}]  {arr.shape}  dtype={arr.dtype}")

data = load_data()

print("Overview of the data")
print_info(data)
print(f"\nTraining datasets: {list(data['train_dataset_names'])}")
print(f"Test datasets:     {list(data['test_dataset_names'])}")
print(f"Training recordings: {len(data['train_dff'])}")
print(f"Test recordings:     {len(data['test_dff'])}")

In [ ]:
# Visualize an example training recording
idx = 0  # change this to browse different recordings
dff = data['train_dff'][idx]
t = data['train_t'][idx]
spikes = data['train_spikes'][idx]
fr = data['train_frame_rates'][idx]
ds_name = str(data['train_datasets'][idx])

fig, axes = plt.subplots(2, 1, figsize=(14, 5))

# Full trace
axes[0].plot(t, dff, 'k', linewidth=0.5)
for sp in spikes:
    axes[0].axvline(sp, color='red', alpha=0.3, linewidth=0.5, ymin=0, ymax=0.1)
axes[0].set_ylabel('ΔF/F')
axes[0].set_title(f'Recording {idx}: {ds_name}  |  {fr:.0f} Hz, {len(dff)} frames, '
                  f'{len(spikes)} spikes, {t[-1]:.0f} s')

# Zoomed window
t0, t1 = 30, 50
m = (t >= t0) & (t <= t1)
axes[1].plot(t[m], dff[m], 'k', linewidth=1)
for sp in spikes[(spikes >= t0) & (spikes <= t1)]:
    axes[1].axvline(sp, color='red', alpha=0.6, linewidth=1)
axes[1].set_ylabel('ΔF/F')
axes[1].set_xlabel('Time (s)')
axes[1].set_title(f'Zoomed: {t0}–{t1} s  (red = electrophysiologically recorded spike times)')

plt.tight_layout()
plt.show()

## Research question

<span style='background: yellow'>
Investigating and Mitigating Domain Shift in Deep Learning-Based Spike Inference Across Calcium Indicators
Does spike inference exhibit asymmetric generalization between GCaMP6f and GCaMP6s?
Can mixed-indicator training reduce this domain shift? How does the model architecture affect spike inference performance?
</span>

Some possible directions (pick one, or formulate your own):

1. **Train a spike inference network and evaluate its generalization.** Build a preprocessing pipeline (resampling, noise augmentation, windowing, target smoothing), train a neural net, and evaluate on the held-out test set. How does performance depend on the training data composition and noise level?

2. **Compare architectures.** Implement at least two different network architectures (e.g. the Cascade default, a deeper/shallower variant, one with skip connections or batch normalization). Which design choices matter most for spike inference performance?

3. **How does temporal smoothing of the ground truth affect inference quality?** Train models with different smoothing kernels (e.g. 25, 50, 100, 200 ms) and evaluate the trade-off between temporal precision and robustness. Optionally compare symmetric vs. causal smoothing kernels.

4. **How well does a network trained on one indicator generalize to another?** Train on GCaMP6f data only and test on GCaMP6s (or vice versa). How much does indicator-specific training improve performance?

Implement all steps necessary to answer your question. Think of: preprocessing, model design, training, evaluation, and visualization. The notebook should be concluded with a summary / conclusions section.

# Delete when finished!


In [ ]:

%load_ext autoreload
%autoreload 2

import utils
import warnings
import importlib
%reload_ext autoreload
warnings.warn("Test ")

importlib.reload(utils)

# preprocessing


###  Frame Rate Matching & Temporal Standardization
To feed a 1D-CNN, every recording must live on the exact same discrete time grid. The raw ΔF/F traces were acquired at ~60 Hz, but exact frame rates vary slightly across recording sessions and datasets. We standardize every trace to a single target frame rate before any further processing.

#### Step 1 — Temporal Standardization & Anti-Aliasing
`standardize_trace()` (`utils.py`)

**Problem**  
Raw ΔF/F traces are not sampled at a perfectly uniform rate, and exact frame rates differ slightly across recording hardware/sessions (~59–61 Hz). A 1D-CNN relies on translationally invariant temporal kernels: if 1 second of data is represented by a different number of frames in different recordings, the network cannot learn a consistent kinetic filter (e.g. for the ~50 ms GCaMP rise time).

**Solution**  
Resample every trace onto one strict, uniform target rate (30 Hz), after first removing frequency content that would otherwise alias into the signal band.

**Why**  
Naively subsampling or linearly interpolating a signal to a lower rate lets high-frequency content (optical shot noise, motion artifacts) fold back ("alias") into the low-frequency calcium-transient band, permanently corrupting the trace. Filtering *before* resampling removes everything above the new Nyquist limit first.

**How**
1. Estimate the original sampling rate from the median dt of the recording's time vector.
2. If downsampling, design an 8th-order Chebyshev Type I low-pass filter (`scipy.signal.cheby1`, `output='sos'`) with a cutoff just below the new Nyquist frequency (15 Hz for a 30 Hz target).
3. Apply the filter with zero phase distortion (`sosfiltfilt`), so the calcium transient's onset timing isn't shifted.
4. Linearly interpolate the filtered trace onto an exact, uniform 30 Hz time grid.

**Why not**
- *`scipy.signal.decimate` / plain `resample`*: reasonable off-the-shelf alternatives, but our target rate (30 Hz) isn't a clean integer factor of every recording's original rate, so an explicit filter + `interp1d` step gives exact control over the output time grid.
- *Filtering unconditionally*: if a recording's original rate is already ≤ the target rate (upsampling), there's no aliasing risk, and requesting a filter cutoff above the input Nyquist would error — so filtering is skipped in that case.
- *Second-order sections (SOS) instead of `b, a` polynomials*: high-order IIR filters expressed as `b, a` coefficients are numerically unstable; SOS form avoids that. `filtfilt` on `b,a` would work too, but `sosfiltfilt` is the more numerically robust equivalent.
### PTemporal Standardization & Anti-Aliasing

#### 1. Standardized Target Frequency
* **The Problem:** The raw $\Delta F/F$ datasets were recorded at ~60 Hz, but individual recording hardware causes slight variations in exact frame rates across the database.
* **The Solution:** We enforce a strict, uniform target sampling rate (e.g., 30 Hz).
* **The "Why":** 1D Convolutional Neural Networks rely on translationally invariant temporal kernels. If 1 second of data is represented by 61 frames in one dataset and 59 frames in another, the CNN cannot learn consistent kinetic features (like the ~50ms GCaMP rise time). 

#### 2. Signal Decimation (Anti-Aliasing)
* **The Problem:** Naively dropping frames (subsampling) or using simple linear interpolation to reduce the frame rate introduces **aliasing**. High-frequency optical shot noise will "fold" back into the lower-frequency biological signal band, permanently corrupting the trace.
* **The Solution:** We use robust downsampling techniques like `scipy.signal.decimate` or `scipy.signal.resample`. 
* **The "Why":** `scipy.signal.decimate` mathematically protects the signal by applying a low-pass Chebyshev Type I (IIR) filter *before* downsampling. This strictly cuts off high frequencies above the new Nyquist limit, ensuring the remaining low-frequency calcium transients are preserved without noise injection. *(Note: `scipy.signal.resample` achieves a similar anti-aliasing effect in the frequency domain via FFT decimate is the mathematically superior choice because it applies the Chebyshev filter and explicitly allows us to enforce zero-phase filtering
#### 2. Anti-Aliasing via Second-Order Sections (sosfiltfilt)
* **The Problem:** Naively reducing the frame rate introduces **aliasing**, where high-frequency optical shot noise folds into the biological signal band. Furthermore, standard `b, a` polynomial filters can become numerically unstable.
* **The Solution:** We apply an explicit low-pass Chebyshev Type I filter (cutoff at the new Nyquist limit of 15 Hz) utilizing **Second-Order Sections (`sosfiltfilt`)**. 
* **Why not a Bandpass filter?** While bandpass filters (500-4000 Hz) are standard for raw electrophysiology to isolate fast voltage transients from slow LFP (as seen in standard spike detection pipelines), optical GCaMP calcium transients are inherently low-frequency events. A bandpass filter would destroy the $\Delta F/F$ signal.
* **Why `sosfiltfilt` over `filtfilt`?** As established in best practices for neural signal processing, extracting filter coefficients as Second-Order Sections (`output='sos'`) prevents the numerical instability and floating-point errors common with high-order `b, a` arrays. The `filtfilt` mechanism ensures the phase remains strictly zero.

#### 3. Phase Preservation (Zero-Phase Filtering)
* **The Problem:** Standard IIR filters (like the default Chebyshev filter) introduce non-linear phase shifts. This means the filter will delay the signal, pushing the $\Delta F/F$ peak slightly forward in time. 
* **The Solution:** We must enforce zero-phase filtering (e.g., using `zero_phase=True` in SciPy's decimate, which applies `filtfilt`—filtering the signal forward, then backward).
* **The "Why":** In spike inference, temporal alignment is everything. If the filter shifts the optical calcium transient forward, it will no longer perfectly align with the ground-truth electrophysiology (the true action potential timestamp). Zero-phase filtering mathematically guarantees that the temporal onset of the biological signal is not artificially shifted.

In [ ]:
from utils import standardize_trace

TARGET_FS = 30.0  # Hz - global standard for the whole project

phase1_data = {}

print(f"Resampling continuous traces to {TARGET_FS} Hz...")

for idx in range(len(data['train_dff'])):
    raw_dff = data['train_dff'][idx]
    raw_t = data['train_t'][idx]

    # Anti-aliased resampling onto the uniform target grid
    dff_res, t_res = standardize_trace(dff=raw_dff, t=raw_t, target_fs=TARGET_FS)

    # Keep everything needed for the next preprocessing steps together,
    # keyed by recording index so nothing gets misaligned downstream.
    phase1_data[idx] = {
        'dff': dff_res,                              # resampled dF/F trace
        'frame_times': t_res,                        # resampled, uniform time vector
        'spikes': data['train_spikes'][idx],         # RAW spike times (point events - not resampled)
        'dataset': str(data['train_datasets'][idx]),
        'filename': str(data['train_filenames'][idx]),
        'orig_frame_rate': data['train_frame_rates'][idx],
    }

print(f"Successfully resampled {len(phase1_data)} recordings.")
print(f"Recording 0 - Original shape: {data['train_dff'][0].shape} "
      f"(~{data['train_frame_rates'][0]:.1f} Hz)")
print(f"Recording 0 - New shape:      {phase1_data[0]['dff'].shape} ({TARGET_FS} Hz)")


#### discrete event binning
Calcium imaging records continuous optical signals, whereas ground truth electrophysiology captures discrete action potentials as a point process. Following the rigorous preprocessing parameters of the Cascade framework (Rupprecht et al., 2021), we mathematically project these exact spike timestamps into discrete counting bins that strictly align with our downsampled $\Delta F/F$ frame rates. 

Furthermore, optical instrumentation inherently introduces micro-delays between biological spiking events and photon integration. To correct this, we utilize a bounded cross-correlation alignment. By isolating a maximum physiological search window ($\pm$ 500 ms), we compute the Pearson correlation coefficient to detect and apply the optimal discrete frame shift ($m^*$). This strict temporal correction guarantees that the fast onset kinetics (~50 ms) of the calcium indicator perfectly synchronize with the electrical ground truth, an absolute prerequisite for accurate 1D-CNN backpropagation.

##### Boundary Counting

##### Count Validation 
### Phase 2: Discrete Event Binning & Physiologically Bounded Lag Correction

**The Objective:** To rigorously map continuous point-process events (action potentials) onto a standardized, discrete temporal lattice without losing event counts, and to correct for hardware-induced temporal misalignments.

**The 'Why' (Rationale for the Poster):**
1. **Conservation of Events:** Standard downsampling or interpolation completely destroys discrete point-process data. By calculating precise temporal boundaries (midpoints between downsampled frames), we guarantee that the integral of the continuous spike train exactly matches the sum of the discrete binned vector. No action potentials are lost or double-counted.
2. **Bounded Cross-Correlation:** Mechanical scanning delays and processing lags cause the optical $\Delta F/F$ signal to slightly trail the electrical ground truth. We correct this by shifting the vectors to maximize their Pearson correlation. Crucially, **this search must be bounded** (strictly to $\pm$ 500 ms). If unbounded, macroscopic baseline drifts (like photobleaching or animal movement) could cause the algorithm to falsely align a spike burst to a noise artifact seconds away. Bounding the shift physically restricts the alignment to the known biological kinematics of GCaMP (~50 ms rise time) plus minor hardware latency.

Discrete Event Binning & Physiologically-Bounded Lag Correction
Calcium imaging records a continuous optical signal; the electrophysiology ground truth is a discrete point process (exact spike times). To train and evaluate a model, every spike must be assigned to exactly one ΔF/F frame, and any fixed hardware timing offset between the two acquisition systems must be corrected.

Step 2 — Discrete Event Binning
bin_discrete_spikes() (utils.py)

Problem
Spike times are continuous; the resampled ΔF/F is discrete. Naive binning (e.g. fixed-width bins with silent edge handling) can lose or double-count events.

Solution
Project each spike time into the bin of its temporally nearest frame, using bin edges placed at the midpoints between consecutive frame timestamps.

Why
Midpoint edges give the least-biased assignment when frame intervals aren't perfectly uniform, and let us explicitly prove no spikes are lost: the sum of the binned counts must equal the number of spikes inside the recording window.

How
Validate that frame times are strictly increasing → build midpoint edges (with extrapolated half-width edges at the two ends) → histogram the spike times into those edges → explicitly check spike conservation → report any spikes that fell outside the recording window.

Why not
A fixed-width/rolling bin (e.g. [frame, frame + dt)) is simpler but biases spikes near a frame boundary toward the earlier frame; midpoint edges instead center each bin on its frame.

Step 3 — Bounded Cross-Correlation Alignment
bounded_cross_correlation_alignment() (utils.py)

Problem
The optical and electrical acquisition systems trigger independently. Mechanical scanning delays and processing latency introduce a small, fixed lag between the two signals, which would otherwise bias any frame-by-frame comparison.

Solution
Search a bounded range of integer-frame shifts (±500 ms) and keep the shift that maximizes the Pearson correlation between the (shifted) ΔF/F trace and the binned spike counts.

Why
The lag reflects a fixed hardware property, not something to fit freely — bounding the search to a physiologically plausible window (based on GCaMP's ~50 ms rise time plus expected hardware latency) prevents the optimizer from "aligning" onto an unrelated, distant feature (e.g. a baseline drift from photobleaching or animal movement).

How
Convert the max lag to frames → shift/truncate/correlate at every candidate lag (skipping flat segments where correlation is undefined) → keep the best-correlating lag → apply it and truncate both series to their overlap.

Why not
An unbounded search is more "flexible" but risks locking onto a spurious, non-physiological shift, especially in short or noisy recordings.

In [ ]:
from utils import bin_discrete_spikes, bounded_cross_correlation_alignment

TARGET_FRAMETE _RA= TARGET_FS   # 30.0 Hz, same standard as Phase 1
MAX_LAG_BOUND_SEC = 0.5          # +/- 500 ms physiological bound

phase2_aligned_data = {}
lag_distribution = []

print(f"Executing Phase 2: Bounded Lag Correction & Binning "
      f"(Target: {TARGET_FRAME_RATE} Hz)...")

for recording_id, recording_data in phase1_data.items():
    spike_times = recording_data['spikes']        # Continuous ground-truth timestamps
    frame_times = recording_data['frame_times']   # Resampled optical timestamps
    dff_trace = recording_data['dff']              # Resampled continuous trace

    # Step 1: Strict boundary binning (raises on any spike-count mismatch)
    binned_spikes = bin_discrete_spikes(
        spike_times=spike_times,
        frame_times=frame_times,
    )

    # Step 2: Bounded cross-correlation alignment
    aligned_dff, aligned_spike_counts, optimal_lag = bounded_cross_correlation_alignment(
        dff_trace=dff_trace,
        binned_spikes=binned_spikes,
        frame_rate=TARGET_FRAME_RATE,
        max_lag_sec=MAX_LAG_BOUND_SEC,
    )

    phase2_aligned_data[recording_id] = {
        'aligned_dff': aligned_dff,
        'aligned_spike_counts': aligned_spike_counts,
        'optimal_lag_frames': optimal_lag,
        'lag_ms': (optimal_lag / TARGET_FRAME_RATE) * 1000,
        'dataset': recording_data['dataset'],
    }
    lag_distribution.append((optimal_lag / TARGET_FRAME_RATE) * 1000)

print(f"Successfully aligned {len(phase2_aligned_data)} recordings.")
print(f"Average temporal lag detected: {np.mean(lag_distribution):.2f} ms")
print(f"Maximum |temporal shift| applied: {np.max(np.abs(lag_distribution)):.2f} ms")


#### Sanity check

Quick visual check that the binned spike counts line up with the ΔF/F rises for one example recording, after alignment.


In [ ]:
# Sanity check: overlay one aligned recording
check_id = 0
rec = phase2_aligned_data[check_id]
t_check = np.arange(len(rec['aligned_dff'])) / TARGET_FRAME_RATE

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t_check, rec['aligned_dff'], 'k', linewidth=0.7, label='aligned ΔF/F')
spike_frames = np.where(rec['aligned_spike_counts'] > 0)[0]
ax.vlines(t_check[spike_frames], 0, rec['aligned_dff'].max(),
          color='red', alpha=0.4, linewidth=0.8, label='binned spikes')
ax.set_xlim(30, 50)
ax.set_xlabel('Time (s)')
ax.set_ylabel('ΔF/F')
ax.set_title(f"Recording {check_id} ({rec['dataset']}) - lag corrected: "
             f"{rec['optimal_lag_frames']} frames ({rec['lag_ms']:.1f} ms)")
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

### Phase 3: Signal Decontamination and Baseline Correction

Before initiating model training, raw optical measurements must be computationally isolated from physical imaging artifacts and slowly varying experimental noise. This pipeline enforces a strict adherence to the kinetics expected by the Cascade framework (Rupprecht et al., 2021).

**1. Neuropil Decontamination**
Two-photon microscopy is susceptible to axial point-spread and out-of-focus fluorescence. Background signal (neuropil) that bleeds into the somatic region of interest (ROI) is mathematically removed via center-surround subtraction:
$$F_{\text{corrected}}(t) = F_{\text{soma}}(t) - \alpha \cdot F_{\text{neuropil}}(t)$$
where $\alpha$ is the empirically determined contamination coefficient (typically ~0.7).

**2. Dynamic Baseline Estimation**
Because the "resting state" of a neuron drifts over time due to photobleaching and indicator washout, a static mean baseline is mathematically invalid. We establish a dynamic baseline $F_0(t)$ using a rolling, asymmetric 8th-percentile filter over a sliding window ($W$):
$$F_0(t) = P_{8}\left( \{ F_{\text{corrected}}(\tau) \mid \tau \in [t - W, t + W] \} \right)$$
The trace is then normalized to relative fluorescence change ($\Delta F/F$):
$$\frac{\Delta F}{F}(t) = \frac{F_{\text{corrected}}(t) - F_0(t)}{|F_0(t)| + \epsilon}$$

**3. The "No-Clipping" Constraint for Noise Integrity**
Neuropil subtraction frequently pushes $F_{\text{corrected}}(t)$ below $F_0(t)$, yielding negative baseline dips. **Crucially, these negative values are strictly preserved and not clipped to zero.** Applying a $\max(x, 0)$ transformation artificially truncates the lower half of the Gaussian/Poisson shot-noise distribution. Preserving the true median absolute deviation of the high-frequency shot noise ($\nu$) is mandatory for Cascade's downstream noise-matching augmentation, ensuring the 1D-CNN properly generalises to novel, noisy datasets.

### Dataset Execution Loop: Binning, Decontamination, and QC

With our rigorous mathematical filtering established in `utils.py`, we now process the raw dataset. This execution loop performs three critical operations per recording:
1. **Phase 2 Binning:** Converts continuous electrophysiological spike timestamps into discrete, frame-matched binned arrays using `np.histogram`. This aligns the electrical ground truth with the optical $\Delta F/F$ traces.
2. **Phase 3 Dynamic Baseline:** Applies our asymmetric percentile filter to correct for low-frequency photobleaching and experimental drift without artificially truncating the Gaussian shot-noise floor.
3. **Phase 3 Quality Control:** Computes the Signal-to-Noise Ratio (SNR) and total spike count. Neurons that are optically unresponsive or contain zero biological action potentials are strictly excluded to prevent gradient washout during CNN training.

In [ ]:

# Initialize storage dictionary for the clean dataset
phase3_clean_data = {}
exclusion_count = 0

print(f"Executing Phase 3: Dynamic Baseline Correction & QC across {len(phase2_aligned_data)} recordings...")

for recording_id, rec in phase2_aligned_data.items():
    aligned_dff = rec['aligned_dff']
    aligned_spikes = rec['aligned_spike_counts']
    dataset_id = rec['dataset']
    
    # ---------------------------------------------------------
    # STEP A: Phase 3 - Dynamic Baseline Correction
    # ---------------------------------------------------------
    dff_clean, f0 = utils.compute_dynamic_dff(
        f_soma=aligned_dff,
        f_neuropil=None, 
        fps=TARGET_FRAME_RATE,
        window_sec=15.0,
        percentile=8
    )
    
    # ---------------------------------------------------------
    # STEP B: Phase 3 - Strict Quality Control
    # ---------------------------------------------------------
    is_valid = utils.robust_quality_control(
        dff=dff_clean,
        spikes=aligned_spikes,
        fps=TARGET_FRAME_RATE,
        min_spikes=5,         # Drop cells with fewer than 5 spikes total
        snr_threshold=2.5     # Drop cells where biological signal is buried in noise
    )
    
    # Filter and Store
    if is_valid:
        phase3_clean_data[recording_id] = {
            'dff_clean': dff_clean,
            'spikes_binned': aligned_spikes,
            'f0_baseline': f0,
            'dataset_id': dataset_id
        }
    else:
        exclusion_count += 1

print("-" * 50)
print("PHASE 3 COMPLETE")
print(f"Successfully processed and retained: {len(phase3_clean_data)} neurons.")
print(f"Excluded due to low SNR or lack of spikes: {exclusion_count} neurons.")
print("-" * 50)

## Target Label Smoothing & Noise Matching

**Objective:** Transform discrete binned spike counts into continuous spike rate targets to optimize the loss landscape for our 1D-CNN. 

**Methodology & Mathematical Justification:**
Training a deep convolutional network to predict strictly sparse, binary vectors (0 or 1) yields an incredibly jagged loss landscape. A temporal prediction error of just one frame ($~33$ ms at 30 Hz) results in a total loss penalty, stripping the network of directional gradients during backpropagation. 

To resolve this, we map the discrete binned spikes into a continuous temporal probability space using a time-symmetric Gaussian convolution:
$K(t) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{t^2}{2\sigma^2}\right)$

Following the standard Cascade framework, the standard deviation $\sigma$ must physically scale with the recording frame rate to preserve physiological kinetics. For our normalized **30 Hz** datasets, we lock $\sigma = 0.05$ seconds. 

By predicting this smoothed spike rate via Mean Squared Error (MSE), the network receives proportional gradient updates for "near-miss" predictions, allowing it to accurately learn indicator rise-and-decay kinetics without becoming trapped in local minima. 

*Note regarding baseline artifacts:* Any negative $\Delta F/F$ dips resulting from neuropil subtraction are preserved in the input matrices. The 1D-CNN's temporal receptive field is mathematically capable of contextualizing these dips, and clipping them would destroy informative local covariance.

Target Label Smoothing & Noise Matching

**Objective:** Transform discrete binned spike counts into continuous spike rate targets to construct a convex loss landscape for 1D-CNN optimization.

**Methodology & Mathematical Justification:**
Training a deep convolutional network to predict strictly sparse, discrete binary vectors yields an extremely jagged loss landscape. Under a pure discrete regime, a temporal prediction error of just one frame (e.g., $~33$ ms at 30 Hz) results in a total loss penalty. This deprives the network of overlapping, directional gradients during backpropagation, causing the model to struggle with temporal alignment.

To resolve this, we map the discrete binned spikes into a continuous temporal probability space using a time-symmetric Gaussian convolution:

$$K(t) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{t^2}{2\sigma^2}\right)$$

Following standard neural decoding practices, the standard deviation $\sigma$ must scale linearly with the recording frame rate to preserve physiological kinetics. For our normalized **30 Hz** datasets, we strictly fix $\sigma$ to **0.05 seconds**.

By predicting this smoothed spike rate via Mean Squared Error (MSE), the network receives proportional, non-zero gradient updates for "near-miss" predictions. This allows the kernels to accurately learn indicator rise-and-decay kinetics without collapsing into local minima.

**Handling Baseline Artifacts:**
During prior somatic decontamination (neuropil subtraction), the $\Delta F/F$ traces frequently dip below zero. **We do not clip these negative baseline dips.** A 1D-CNN with a wide temporal receptive field mathematically contextualizes these negative dips as valid features of indicator decay and surrounding neuropil dynamics. Clipping them with a zero-floor would permanently destroy biological covariance in the input matrix.

In [ ]:
# --- ADD TO spike_inference.ipynb ---
import numpy as np
import matplotlib.pyplot as plt
import utils  # Make sure your smooth_spike_train function is saved here

# TARGET PARAMETERS (Cascade Standard)
# Relying directly on the standard you defined in Phase 2
SIGMA_SECONDS = 0.05

print(f"Executing Phase 4: Target Label Smoothing (sigma={SIGMA_SECONDS}s)...")

# 1. Apply smoothing strictly PER RECORDING to prevent boundary leakage
for recording_id, rec_data in phase2_aligned_data.items():
    # Extract the discrete aligned bins you generated in Phase 2
    binned_spikes = rec_data['aligned_spike_counts']
    
    # Smooth the spikes
    smoothed_rates = utils.smooth_spike_train(
        spike_counts=binned_spikes, 
        target_fs=TARGET_FRAME_RATE, 
        sigma_sec=SIGMA_SECONDS
    )
    
    # Store the continuous targets back into your master dictionary
    phase2_aligned_data[recording_id]['smoothed_spike_rates'] = smoothed_rates

print("Phase 4 smoothing complete. Continuous targets added to dictionary.")


## Sanity check 

In [ ]:

# 2. Visual Sanity Check (Best Practice: Build Data Intuition)
# Dynamically select the first recording in your dictionary for visualization
example_rec_id = list(phase2_aligned_data.keys())[0]
example_data = phase2_aligned_data[example_rec_id]

# Slice the data for a 10-second plot
frames_to_plot = int(10 * TARGET_FRAME_RATE) 
time_axis = np.arange(frames_to_plot) / TARGET_FRAME_RATE

binned_plot = example_data['aligned_spike_counts'][:frames_to_plot]
smoothed_plot = example_data['smoothed_spike_rates'][:frames_to_plot]

# Generate Publication-Ready Plot
fig, ax1 = plt.subplots(figsize=(10, 3), dpi=120)

# Plot discrete spikes as a stem plot on a secondary y-axis for clarity
ax2 = ax1.twinx()
ax2.stem(time_axis, binned_plot, 
         linefmt='grey', markerfmt='k|', basefmt=' ', label='Discrete Bins')
ax2.set_ylabel('Discrete Count', color='grey')
ax2.set_ylim(0, max(binned_plot) + 1) 
ax2.tick_params(axis='y', labelcolor='grey')

# Plot the smoothed continuous target
ax1.plot(time_axis, smoothed_plot, 
         color='blue', linewidth=2, label=rf'Smoothed Rate ($\sigma$={SIGMA_SECONDS}s)')
ax1.set_xlabel('Time (seconds)')
ax1.set_ylabel('Spike Rate (Hz)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

plt.title(f'Phase 4: Transformation of Discrete Bins to Continuous Target\n(Recording ID: {example_rec_id})')
fig.tight_layout()
plt.show()

## Strict Dataset Partitioning


efore generating sliding windows for batching, the dataset must be rigidly split.Total Isolation: The data must be partitioned at the whole-recording or dataset level to ensure that sliding temporal windows do not leak across train and test boundaries.Held-out Evaluation: Test recordings (test_*) must remain entirely untouched during this preprocessing phase to guarantee a valid final evaluation.  

#### Physiologically bounded temporal alignment

##### Cross-Correlation Peak Search

##### constrained Windowing

##### array shifting 


Compute a per-recording noise metric (Cascade defines one from the frame-to-frame ΔF/F fluctuation, normalized by frame rate) and note it per recording —  check whether "domain shift" is really indicator-shift or just noise-level shift, since 6f and 6s recordings may differ in SNR as well as kinetics.

z-score or robust scale 

windowing

split by recording

assemble pools

## model design 

## training

#### train: 6f only


### train: 6s only

### train: mixed

## evaluation

## visualisation

# summary / conclusion